# Random Forest — Quality & Commercial Score

Trains a `RandomForestRegressor` for each target. No feature scaling needed.
Saves the fitted models and a markdown report to `models/random_forest/`.

In [1]:
# ── Constants ──────────────────────────────────────────────────────────────
import os

TRAIN_RATIO = float(os.environ.get("LUDOMETRICS_TRAIN_RATIO", "0.80"))
TEST_RATIO = round(1.0 - TRAIN_RATIO, 10)
RANDOM_STATE = 42
N_ESTIMATORS = 200
MAX_DEPTH = 20
TOP_N_FEATURES = 10

SPLIT_LABEL = f"{int(TRAIN_RATIO * 100)}_{int(TEST_RATIO * 100)}"
SPLIT_DISPLAY = f"{int(TRAIN_RATIO * 100)}/{int(TEST_RATIO * 100)}"

In [2]:
import sys
import time

sys.path.insert(0, "..")

import joblib
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from utils.train_utils import top_feature_importances, update_results_table

## 1. Load data

In [3]:
df = pd.read_csv("../data/games_processed.csv")
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

Loaded 21,925 rows × 403 columns


## 2. Define feature columns

In [4]:
DROP_COLS = ["BGGId", "quality_score", "commercial_score"]
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
TARGETS = ["quality_score", "commercial_score"]

X = df[FEATURE_COLS]
print(f"Features: {len(FEATURE_COLS)} columns")

Features: 400 columns


## 3. Train / test split

In [5]:
X_train, X_test = train_test_split(X, test_size=TEST_RATIO, random_state=RANDOM_STATE)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Train: 19,732  |  Test: 2,193


## 4. Train one model per target

In [6]:
results = {}

for target in TARGETS:
    y_train = df.loc[X_train.index, target]
    y_test = df.loc[X_test.index, target]

    model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    t0 = time.monotonic()
    model.fit(X_train, y_train)
    elapsed = time.monotonic() - t0

    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    results[target] = {"model": model, "rmse": rmse, "r2": r2, "elapsed": elapsed}
    print(f"{target:20s}  RMSE={rmse:.4f}  R²={r2:.4f}")

quality_score         RMSE=2.4621  R²=0.6073


commercial_score      RMSE=9.5172  R²=0.5713


## 5. Save models

In [7]:
MODEL_DIR = Path("../results/models/random_forest")

for target in TARGETS:
    out_path = MODEL_DIR / f"{target}_{SPLIT_LABEL}.pkl"
    joblib.dump(results[target]["model"], out_path)
    print(f"Saved {out_path}")

Saved ../results/models/random_forest/quality_score_90_10.pkl


Saved ../results/models/random_forest/commercial_score_90_10.pkl


## 6. Write report

In [8]:
top_features = top_feature_importances(
    results["quality_score"]["model"], FEATURE_COLS, n=TOP_N_FEATURES
)

metrics = {
    target: {"rmse": results[target]["rmse"], "r2": results[target]["r2"]}
    for target in TARGETS
}

print()

RESULTS_DIR = Path("../results")

for target in TARGETS:
    elapsed = results[target]["elapsed"]
    minutes, seconds = divmod(elapsed, 60)
    time_label = (
        f"{int(minutes)}m {seconds:.1f}s" if minutes >= 1 else f"{seconds:.1f}s"
    )
    update_results_table(
        RESULTS_DIR / f"{target}.md",
        algorithm="Random Forest (RandomForestRegressor)",
        split=SPLIT_DISPLAY,
        train_size=len(X_train),
        test_size=len(X_test),
        training_time=time_label,
        rmse=results[target]["rmse"],
        r2=results[target]["r2"],
    )
    print(f"Updated results/{target}.md")


Updated results/quality_score.md
Updated results/commercial_score.md
